# Flood ground-truth: groundsource events + NWS flood warnings

Two complementary flood layers, loaded, summarized, and merged into one harmonized
GeoDataFrame for pairing with the GOES imagery:

1. **groundsource** (`data/raw/groundsource_2026.parquet`) — observed flood
   *extents* (polygons) with a start/end date. ~2.65M rows, global.
2. **flood warnings** (`data/flood_warnings/flood_warnings_conus.parquet`) — NWS
   forecaster-issued **Flash Flood (FF)** + **Areal Flood (FA)** *Warning* polygons,
   CONUS 2019–2026, built by `src/download_flood_data.py`.

They differ in nature — observed extent vs. issued warning — so we keep a `source`
column rather than blending them blindly.

In [ ]:
import geopandas as gpd
import pandas as pd

pd.set_option("display.max_columns", None)

GROUNDSOURCE = "data/raw/groundsource_2026.parquet"
WARNINGS = "data/flood_warnings/flood_warnings_conus.parquet"

# CONUS lon/lat box + study window (matches the warnings coverage)
CONUS = (-125, -66, 24, 49)
START = "2019-01-01" 

## 1. Load

In [ ]:
# --- groundsource: observed flood extents ---
gs = gpd.read_parquet(
    GROUNDSOURCE,
    columns=["uuid", "area_km2", "start_date", "end_date", "geometry"],
)
gs["start_date"] = pd.to_datetime(gs["start_date"])
gs["end_date"] = pd.to_datetime(gs["end_date"])

# restrict to the CONUS + 2019-on study window (aligns with the warnings)
gs = gs.cx[CONUS[0]:CONUS[1], CONUS[2]:CONUS[3]]
gs = gs[gs["start_date"] >= START].copy()
print(f"groundsource (CONUS, >= {START}): {len(gs):,} rows")
gs.head(3)

In [ ]:
# --- NWS flood warnings (already CONUS 2019-2026) ---
fw = gpd.read_parquet(WARNINGS)
# IEM timestamps are ISO 'Z' (UTC); make them tz-naive UTC to match groundsource
fw["issue"] = pd.to_datetime(fw["issue"], utc=True).dt.tz_localize(None)
fw["expire"] = pd.to_datetime(fw["expire"], utc=True).dt.tz_localize(None)
print(f"flood warnings: {len(fw):,} rows")
fw.head(3)

## 2. Summarize

In [ ]:
def summarize(gdf, name, id_col, start_col, end_col, area_col):
    """One-line-per-stat summary of a flood GeoDataFrame."""
    print(f"=== {name} ===")
    print(f"rows            : {len(gdf):,}")
    print(f"unique ids      : {gdf[id_col].nunique():,}")
    print(f"date range      : {gdf[start_col].min():%Y-%m-%d} "
          f"-> {gdf[end_col].max():%Y-%m-%d}")
    dur = (gdf[end_col] - gdf[start_col]).dt.days
    print(f"duration (days) : median {dur.median():.0f}, mean {dur.mean():.1f}, "
          f"max {dur.max():.0f}")
    a = gdf[area_col]
    print(f"area_km2        : sum {a.sum():,.0f}, median {a.median():.2f}, "
          f"mean {a.mean():.1f}, max {a.max():,.0f}")
    print(f"geometry types  : {gdf.geometry.geom_type.value_counts().to_dict()}")
    print(f"bounds (lon/lat): {gdf.total_bounds.round(2).tolist()}")
    print()


summarize(gs, "groundsource (observed extents)", "uuid",
          "start_date", "end_date", "area_km2")
summarize(fw, "NWS flood warnings", "key", "issue", "expire", "area_iem")

In [ ]:
# warnings: breakdown by phenomena, year, and polygon source
print("by phenomena:", fw.phenomena.value_counts().to_dict())
print("polygon source:", fw.polygon_source.value_counts().to_dict())
fw.assign(year=fw.issue.dt.year).groupby(["year", "phenomena"]).size().unstack(fill_value=0)

In [ ]:
# groundsource: events per year (by start_date)
gs.assign(year=gs.start_date.dt.year).groupby("year").size()

## 3. Harmonized unified frame

One schema across both sources so they can be filtered, joined to GOES dates, and
overlaid together. Columns:

| column | groundsource | flood warning |
|---|---|---|
| `source` | `"groundsource"` | `"ff_warning"` / `"fa_warning"` |
| `event_id` | `uuid` | `key` (`year_wfo_ph_etn`) |
| `phenomena` | `"OBS"` | `FF` / `FA` |
| `issue_date` | `start_date` | `issue` |
| `expire_date` | `end_date` | `expire` |
| `area_km2` | `area_km2` | `area_iem` |
| `geometry` | polygon | polygon |

In [ ]:
COLS = ["source", "event_id", "phenomena", "issue_date", "expire_date",
        "area_km2", "geometry"]

gs_u = gpd.GeoDataFrame({
    "source": "groundsource",
    "event_id": gs["uuid"].astype(str),
    "phenomena": "OBS",
    "issue_date": gs["start_date"],
    "expire_date": gs["end_date"],
    "area_km2": gs["area_km2"],
    "geometry": gs.geometry,
}, crs=gs.crs)[COLS]

fw_u = gpd.GeoDataFrame({
    "source": fw["phenomena"].map({"FF": "ff_warning", "FA": "fa_warning"}),
    "event_id": fw["key"],
    "phenomena": fw["phenomena"],
    "issue_date": fw["issue"],
    "expire_date": fw["expire"],
    "area_km2": fw["area_iem"],
    "geometry": fw.geometry,
}, crs=fw.crs)[COLS]

floods = gpd.GeoDataFrame(
    pd.concat([gs_u, fw_u], ignore_index=True), crs="EPSG:4326"
)
floods["issue_day"] = floods["issue_date"].dt.normalize()   # date key for GOES join
print(f"unified: {len(floods):,} rows")
print("by source:", floods.source.value_counts().to_dict())
floods.head(3)

### Persist the unified frame

Save to `data/flood_warnings/floods_unified.parquet` so other notebooks (e.g. `clouds_vs_floods.ipynb`) can overlay it without rebuilding.

In [ ]:
UNIFIED_OUT = "data/flood_warnings/floods_unified.parquet"
floods.to_parquet(UNIFIED_OUT)
print(f"wrote {UNIFIED_OUT}  ({len(floods):,} rows)")

In [ ]:
# sanity: schema, null geometry, date span
print(floods.dtypes)
print("\nnull/empty geometry:", int(floods.geometry.isna().sum()),
      "/", int(floods.geometry.is_empty.sum()))
print("issue_date span:", floods.issue_date.min(), "->", floods.issue_date.max())
floods.describe(include="all").T

### Optional: overlay a sample on a map

`.explore()` puts a sample of both sources on one interactive map (warnings vs.
observed extents by colour).

In [ ]:
sample = floods.sample(10000, random_state=69)
sample.explore(
    column="source",
    tooltip=["source", "phenomena", "issue_date", "expire_date", "area_km2"],
    popup=True, cmap="viridis", style_kwds={"fillOpacity": 0.4},
)